# Set up Zerobus ingestion table and Gmail secrets

Run this notebook once before deploying the CDP data pipeline.

It covers two prerequisites:

1. Create the Unity Catalog table that the Zerobus Google Tag Manager connector writes into.
2. Create and validate the Databricks secret used by the Gmail email sinks.

The table schema is copied from `INSTALL.md` in the Zerobus community connector, lines 142-151.

## 1. Configure target objects

The defaults match `data_pipeline/databricks.yml`:

- Catalog: `gtm_zerobus_demo_catalog`
- Schema: `landing`
- Table: `gtm_events`

Change the widget values if you deploy the bundle with different `table_catalog` or `table_schema` variables.

In [ ]:
dbutils.widgets.text("catalog", "gtm_zerobus_demo_catalog", "Unity Catalog catalog")
dbutils.widgets.text("schema", "landing", "Unity Catalog schema")
dbutils.widgets.text("table", "gtm_events", "Zerobus destination table")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
TABLE = dbutils.widgets.get("table").strip()

if not CATALOG or not SCHEMA or not TABLE:
    raise ValueError("catalog, schema, and table widgets must all be set.")

TABLE_FQN = f"`{CATALOG}`.`{SCHEMA}`.`{TABLE}`"

print(f"Target table: {TABLE_FQN}")

## 2. Create the Zerobus table

Zerobus writes one row per GTM event into this table. The structured columns capture request metadata, and `eventData` stores the full GTM event payload as a JSON string.

Reference DDL from `INSTALL.md`:

```sql
CREATE TABLE your_catalog.your_schema.gtm_events (
  ingestion_time BIGINT,
  gtm_container_id STRING,
  event_name STRING,
  request_path STRING,
  request_method STRING,
  query_string STRING,
  visitor_region STRING,
  eventData STRING  -- JSON blob of full event data
);
```

The cell below creates the catalog and schema if they do not exist, then creates the table if needed.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {TABLE_FQN} (
      ingestion_time BIGINT,
      gtm_container_id STRING,
      event_name STRING,
      request_path STRING,
      request_method STRING,
      query_string STRING,
      visitor_region STRING,
      eventData STRING COMMENT 'JSON blob of full event data'
    )
    USING DELTA
    """
)

print(f"Created or verified {TABLE_FQN}")

In [ ]:
display(spark.sql(f"DESCRIBE TABLE {TABLE_FQN}"))

## 3. Grant connector permissions

The Zerobus connector authenticates to Databricks with a service principal. Grant that principal access to the destination table.

Replace `<service-principal-application-id-or-name>` with the principal configured in the Zerobus connector.

```sql
GRANT USE CATALOG ON CATALOG gtm_zerobus_demo_catalog TO `<service-principal-application-id-or-name>`;
GRANT USE SCHEMA ON SCHEMA gtm_zerobus_demo_catalog.landing TO `<service-principal-application-id-or-name>`;
GRANT SELECT, MODIFY ON TABLE gtm_zerobus_demo_catalog.landing.gtm_events TO `<service-principal-application-id-or-name>`;
```

If you changed the widgets above, use your selected catalog, schema, and table names in the grants.

## 4. Create Gmail secrets

The pipeline email sinks send welcome, purchase, and abandoned-cart emails through Gmail SMTP. The current pipeline code expects:

- Gmail address: `nikolaos.servos@gmail.com`
- Secret scope: `cdp_demo`
- Secret key: `gmail_app_password`

Create a Gmail App Password from the Gmail account first. Gmail requires 2-Step Verification to be enabled before App Passwords are available.

Run these commands from your laptop or Databricks CLI environment, not inside this notebook, because `put-secret` prompts for the secret value interactively:

```bash
databricks secrets create-scope cdp_demo -p FEVM
databricks secrets put-secret cdp_demo gmail_app_password -p FEVM
# Paste the 16-character Gmail App Password when prompted, then press Ctrl-D.
```

Optional, if you later update the pipeline code to read the Gmail address from a secret as well:

```bash
printf '%s' 'nikolaos.servos@gmail.com' | databricks secrets put-secret cdp_demo gmail_address -p FEVM
```

In [ ]:
dbutils.widgets.text("secret_scope", "cdp_demo", "Gmail secret scope")
dbutils.widgets.text("gmail_app_password_key", "gmail_app_password", "Gmail app password key")

SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
GMAIL_APP_PASSWORD_KEY = dbutils.widgets.get("gmail_app_password_key").strip()

gmail_app_password = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key=GMAIL_APP_PASSWORD_KEY,
)

# Do not print the secret. This only confirms that the key resolves.
print(
    {
        "scope": SECRET_SCOPE,
        "key": GMAIL_APP_PASSWORD_KEY,
        "secret_loaded": bool(gmail_app_password),
        "gmail_address_used_by_pipeline": "nikolaos.servos@gmail.com",
    }
)

## 5. Quick readiness checklist

- `gtm_zerobus_demo_catalog.landing.gtm_events` exists and `DESCRIBE TABLE` shows the expected columns.
- The Zerobus service principal has `USE CATALOG`, `USE SCHEMA`, `SELECT`, and `MODIFY` on the destination table.
- The Databricks secret `cdp_demo/gmail_app_password` exists and the validation cell above returns `secret_loaded: true`.
- The Gmail account has 2-Step Verification enabled and the secret value is a Gmail App Password, not the normal account password.

After these checks pass, deploy or restart the DLT pipelines so the email sinks can read the secret at startup.